In [1]:
import numpy as np
import pandas as pd
import ast

movies = pd.read_csv("./tmdb_5000_movies.csv")
credits = pd.read_csv("./tmbd_5000_credits.csv")

movies.head(1)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [2]:
credits.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [3]:
print(credits.shape)
print(movies.shape)

(4803, 4)
(4803, 20)


In [4]:
movies = movies.merge(credits,on="title")
movies.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count', 'movie_id', 'cast', 'crew'],
      dtype='object')

In [5]:
# delect unnecessary columns :-
# take :- genres,id, keywords,title, overview, cast
movies = movies.drop(columns=['budget','homepage','original_language','original_title','popularity','production_companies','release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'vote_average','vote_count', 'movie_id','production_countries'])



In [6]:
#handle missing value 
# there is null value in overview column so drop that whole row.
movies.dropna(inplace=True)


In [7]:
movies.head(1)
movies.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [8]:
# GENRES COLUMN
# just get only name from genres column 
def convert(obj):
    name = []
    for i in ast.literal_eval(obj):
        name.append(i['name'])
    return name


In [9]:
# CAST COLUMN 

def cast_convert(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
         if counter != 3: 
          L.append(i['character'])
          counter+=1
         else:
            break
    return L

In [10]:
temp = movies['crew'][0]

In [11]:
# CREW COLUMN
def fatch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
      if i['job'] == 'Director':
         L.append(i['name'])
         break
    return L         
movies['crew'] = movies['crew'].apply(fatch_director)
        

In [12]:
movies['crew'][0]

['James Cameron']

In [13]:
# making easy all columns

movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(cast_convert)

In [14]:
# Overview column is a string and we need to make it list becouse of rest of the column are list 

movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [15]:
# now remove space btw two words that would be good for our recommendation system

movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ","") for i in x])





In [16]:
# create a single column and add all the columns 
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [17]:
# create a new data frame
new_df = movies[["id",'title','tags']]


In [18]:
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))
new_df.head()

C:\Users\HP\AppData\Local\Temp\ipykernel_964\2951490284.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))


,id,title,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...
4,49529,John Carter,"John Carter is a war-weary, former military ca..."


In [19]:
# make it all in lower case 
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

C:\Users\HP\AppData\Local\Temp\ipykernel_964\761566873.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())


In [20]:

# KEYWORDS COLUMN
movies.head(1)


,genres,id,keywords,overview,title,cast,crew,tags
0,"[Action, Adventure, Fantasy, ScienceFiction]",19995,"[cultureclash, future, spacewar, spacecolony, ...","[In, the, 22nd, century,, a, paraplegic, Marin...",Avatar,"[JakeSully, Neytiri, Dr.GraceAugustine]",[JamesCameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."


In [21]:

# KEYWORDS COLUMN
# just get only name from genres column 

